In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
from CompoTree import ComponentTree, Radicals, TSVariants, CharLexicon
from gensim.models.keyedvectors import KeyedVectors
import numpy as np
import numpy.linalg as la

In [ ]:

kv = KeyedVectors.load("../data/fasttext-zh-sw/fasttext-zh-sw.kv")
kv.init_sims()

In [ ]:
ctree = ComponentTree.load()
radicals = Radicals.load()
tsvars = TSVariants.load()
lexicon = CharLexicon.load()


In [ ]:
ctree.query("?", use_flag="first")[0].children

In [ ]:
tsvars.convert("?")

In [ ]:
from collections import defaultdict
chars_set = set()
compo_set = set()
decompo_set = set()
compo_items = set()
for word in lexicon:  
    if word not in kv.vocab:
        continue
        
    try:
        ortho_x = ctree.query(word, 
                          max_depth=1, 
                          use_flag="shortest")[0]
    except:
        print(word)
        continue
            
    if isinstance(ortho_x, str):
        compos = [ortho_x]
    else:
        compos = [str(x) for x in 
                  ortho_x.components(use_flag="shortest")]
        
    compo_seq = tuple(sorted(compos))     
    if compo_seq not in compo_items and len(compos) == 2:
        compo_items.add(compo_seq)
    else:        
        continue
    
    chars_set.add(word)    
    if tsvars.is_simplified(word):
        compos = [x+"_s" for x in compos]
    else:
        compos = [x+"_t" for x in compos]
       
    compo_set |= set(compos)
    decompo_set |= set((word, str(compo_x))
                       for compo_x in compos)    

In [ ]:
len(chars_set), len(compo_set), len(decompo_set)

In [ ]:
sorted(list(decompo_set), key=lambda x: x[0])[:20]

## Component Matrix

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [ ]:
char_enc = LabelEncoder()
compo_enc = LabelEncoder()
char_enc.fit(list(chars_set))
compo_enc.fit(list(compo_set))

In [ ]:
n_char = len(char_enc.classes_)
n_compo = len(compo_enc.classes_)

In [ ]:
decompo_matrix = np.zeros((n_char, n_compo), dtype=np.int32)
decompo_matrix.shape

In [ ]:
for item_x in decompo_set:
    char_idx = char_enc.transform([item_x[0]])[0]
    compo_idx = compo_enc.transform([item_x[1]])[0]
    decompo_matrix[char_idx, compo_idx] = 1

In [ ]:
aa = LabelEncoder()
aa.fit(["a", "b", "c"])

In [ ]:
sel_compo_idx = decompo_matrix.sum(axis=0)>1
sel_compo_enc = LabelEncoder()
sel_compo_enc.fit(compo_enc.classes_[sel_compo_idx])
decompo_matrix_reduced = decompo_matrix[:, sel_compo_idx]
print(decompo_matrix_reduced.shape)
la.matrix_rank(decompo_matrix_reduced)

In [ ]:
## Character semantic vector
word_kv_idxs = []
for word in char_enc.classes_:
    word_kv_idxs.append(kv.vocab[word].index)
char_ft = kv.vectors_norm[word_kv_idxs, :]
char_ft.shape

## Component semantic matrix
$$
C_{k\times d}: \textrm{Component semantic matrix, k components $\times$ d vector size }\\
Z_{n\times d}: \textrm{Character semantic matrix, n characters $\times$ d vector size }\\
D_{n\times k}: \textrm{Decomposition matrix, n characters $\times$ k components}\\
Z = D\cdot C\\
C = (D^\top D)^{-1}D^\top Z
$$

In [ ]:
import numpy as np
import numpy.linalg as la

D = decompo_matrix_reduced
Z = char_ft
print("D, Z: ", D.shape, Z.shape)
compo_ft = la.inv(D.T.dot(D)).dot(D.T).dot(Z)

In [ ]:
Zhat = D.dot(compo_ft)
R = Z.dot(Zhat.T)

In [ ]:
def query_component(compo, is_trad=True):
    if is_trad is not None:
        compo = compo + ("_t" if is_trad else "_s")
    compo_vec = compo_ft[sel_compo_enc.transform([compo])[0]]
    return compo_vec

In [ ]:
list(zip(kv.similar_by_vector(query_component("?", is_trad=True)),
    kv.similar_by_vector(query_component("?", is_trad=False))))

In [ ]:
{v: k for k, v in radicals.ts_radicals.items()}["?"]

In [ ]:
tsvars.is_traditional("?")

In [ ]:
list(zip(kv.similar_by_vector(query_component("?", is_trad=True)),
    kv.similar_by_vector(query_component("?", is_trad=False))))

In [ ]:
list(zip(kv.similar_by_vector(query_component("?", is_trad=True)),
    kv.similar_by_vector(query_component("?", is_trad=False))))

In [ ]:
list(zip(kv.similar_by_vector(query_component("?", is_trad=True)),
    kv.similar_by_vector(query_component("?", is_trad=False))))

In [ ]:
simp_radicals = [x+"_s" for x in radicals.ts_radicals.keys()]
trad_radicals = [x+"_t" for x in radicals.ts_radicals.values()]
targets = [("?_s", "?_t"), ("?_s", "?_t"), ("?_s", "?_t")]

rois = []
for s, t in targets:
    if s in sel_compo_enc.classes_ and \
       t in sel_compo_enc.classes_:
        rois += [s, t]
print("rois count: ", len(rois))
for srad, trad in zip(simp_radicals, trad_radicals):
    if srad in sel_compo_enc.classes_ and \
       trad in sel_compo_enc.classes_:
        rois += [srad, trad]
print("rois count: ", len(rois))

In [ ]:
roi_vectors = np.zeros((len(rois), compo_ft.shape[1]), dtype=np.float)
for roi_idx, roi_x in enumerate(rois):
    roi_vectors[roi_idx] = query_component(roi_x, is_trad=None)

In [ ]:
roi_vectors.shape

In [ ]:
from sklearn.decomposition import PCA
roi_pca = PCA(3)
roi_2d = roi_pca.fit_transform(roi_vectors)

In [ ]:
from matplotlib import font_manager

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "HanaMinA" 
plt.figure(figsize=(12,6))
idx_iter = iter(range(len(rois)))
dim_x = 0
dim_y = 2
for s_idx, t_idx in zip(*([idx_iter]*2)):
    s_compo = rois[s_idx].split("_")[0]
    t_compo = rois[t_idx].split("_")[0]
    
    text_args = dict(fontsize=14)
    if (rois[s_idx], rois[t_idx]) in targets:
        text_args["fontsize"]=20
    
    plt.text(roi_2d[s_idx, dim_x], roi_2d[s_idx, dim_y], 
             s_compo, color="red", **text_args)
    plt.text(roi_2d[t_idx, dim_x], roi_2d[t_idx, dim_y], 
             t_compo, color="blue", **text_args)
    plt.plot(roi_2d[[s_idx, t_idx], dim_x], roi_2d[[s_idx, t_idx], dim_y], color='#AAAAAA')
coord_max = roi_2d.max(axis=0)+0.1
coord_min = roi_2d.min(axis=0)-0.1
plt.xlim(coord_min[dim_x], coord_max[dim_x])
plt.ylim(coord_min[dim_y], coord_max[dim_y])
plt.xlabel(f"PCA-{dim_x}")
plt.ylabel(f"PCA-{dim_y}")
from matplotlib.patches import Patch
h_legends = [Patch(facecolor="red", label="simplified"),
             Patch(facecolor="blue", label="traditional")]
plt.legend(handles=h_legends, loc="upper right")

## Output component vectors

In [ ]:
import pickle
out_data = {
    "compovec": compo_ft,
    "components": sel_compo_enc.classes_.tolist()
}
with open("../data/exp/compo_vectors.pkl", "wb") as fout:
    pickle.dump(out_data, fout)